# Train SFW-SwinCBM on Google Colab
Notebook nay clone repo, mount Google Drive, dung Defactify Hugging Face streaming mac dinh va chay `src.model.train`.


## 1. Check GPU
Runtime > Change runtime type > GPU. T4 là đủ cho cấu hình mặc định.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

## 2. Mount Google Drive
Drive dung de luu checkpoint/output va cache nhe cho Hugging Face. Dataset mac dinh se stream tu Hugging Face, khong can tai full ve Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone repo branch ai_core
Cell nay dung Python thuan de tranh loi current directory bi xoa hoac path bi thanh `/contentn`.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/chtr302/ai_generated_image_detection.git'
BRANCH = 'ai_core'
REPO_DIR = Path('/content/ai_generated_image_detection')

# Reset cwd ve /content truoc khi xoa repo cu.
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
print('cwd:', Path.cwd())
subprocess.run(['git', 'branch', '--show-current'], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 4. Install dependencies
Colab thuong co san torch/torchvision. Cell nay chi cai them thu vien doc Hugging Face dataset va Pillow neu thieu.


In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/ai_generated_image_detection')
if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repo chua clone thanh cong: {REPO_DIR}')
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

!python -m pip install -q datasets pillow


## 5. Verify repo version
Cell nay kiem tra branch `ai_core` tren GitHub da co train script moi ho tro Hugging Face streaming chua. Neu fail, can push code local len GitHub roi restart runtime.


In [ ]:
import subprocess
import sys

help_result = subprocess.run(
    [sys.executable, '-m', 'src.model.train', '--help'],
    check=True,
    capture_output=True,
    text=True,
)
required_args = ['--hf-dataset', '--hf-shuffle-buffer', '--max-train-steps']
missing = [arg for arg in required_args if arg not in help_result.stdout]
if missing:
    subprocess.run(['git', 'branch', '--show-current'], check=False)
    subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
    raise RuntimeError(
        'Repo tren Colab dang la code cu, thieu args: ' + ', '.join(missing) + '\n'
        'Hay commit va push cac thay doi ai_core len GitHub, sau do Runtime > Restart runtime va chay lai notebook.'
    )
print('OK: train.py supports Hugging Face streaming args')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 6. Set data and output paths
Mac dinh notebook dung Defactify tren Hugging Face:

```text
Rajarshi-Roy-research/Defactify_Image_Dataset
Image -> anh
Label_A -> 0 real, 1 AI-generated
Label_B/Caption -> metadata
```

Neu muon train bang folder local trong Drive, gan `DATA_ROOT = Path('/content/drive/MyDrive/ai_data')`.


In [ ]:
from pathlib import Path

# Mac dinh: stream Defactify tu Hugging Face, khong tai full dataset truoc khi train.
HF_DATASET = 'Rajarshi-Roy-research/Defactify_Image_Dataset'
HF_CACHE_DIR = Path('/content/drive/MyDrive/hf_cache')
HF_SHUFFLE_BUFFER = 10000
HF_NO_STREAMING = False

# Neu da co data local tren Drive thi set DATA_ROOT, neu khong thi de None.
DATA_ROOT = None  # vi du: Path('/content/drive/MyDrive/ai_data')
OUTPUT_DIR = Path('/content/drive/MyDrive/ai_detector_outputs')

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def build_data_arg():
    if DATA_ROOT is not None:
        if not Path(DATA_ROOT).exists():
            raise FileNotFoundError(f'DATA_ROOT khong ton tai: {DATA_ROOT}')
        return f'--data-root {DATA_ROOT}'

    args = f'--hf-dataset {HF_DATASET} --hf-cache-dir {HF_CACHE_DIR} --hf-shuffle-buffer {HF_SHUFFLE_BUFFER}'
    if HF_NO_STREAMING:
        args += ' --hf-no-streaming'
    return args

print('HF_DATASET:', HF_DATASET)
print('HF_CACHE_DIR:', HF_CACHE_DIR)
print('HF_NO_STREAMING:', HF_NO_STREAMING)
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('data args:', build_data_arg())


## 7. Quick tests
Chay unit test truoc de bat loi import/data API. Cell nay khong tai dataset that.


In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python -B -m unittest discover -s src/tests -p "test_*.py" -v


## 8. Smoke train
Chay 10 batch dau voi image size nho de kiem tra Hugging Face streaming, transform, forward, loss va checkpoint. Neu cell nay pass thi moi train full.


In [ ]:
SMOKE_IMAGE_SIZE = 128
SMOKE_BATCH_SIZE = 2
SMOKE_STEPS = 10

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size {SMOKE_IMAGE_SIZE} \
  --batch-size {SMOKE_BATCH_SIZE} \
  --epochs 1 \
  --grad-accum 1 \
  --nec 10 \
  --amp none \
  --num-workers 0 \
  --max-train-steps {SMOKE_STEPS}


## 9. Train full
Giam `BATCH_SIZE` xuong 4 neu T4 bi OOM. Streaming dataset khong co tong so step co dinh, nen log se hien `?/epoch`.


In [ ]:
IMAGE_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 20
GRAD_ACCUM = 2
NEC = 10
AMP = 'fp16'
NUM_WORKERS = 2
MAX_TRAIN_STEPS = 0  # dat >0 neu muon gioi han batch moi epoch

data_arg = build_data_arg()
step_arg = f'--max-train-steps {MAX_TRAIN_STEPS}' if MAX_TRAIN_STEPS else ''
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size {IMAGE_SIZE} \
  --batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --grad-accum {GRAD_ACCUM} \
  --nec {NEC} \
  --amp {AMP} \
  --num-workers {NUM_WORKERS} \
  {step_arg}


## 10. Resume training
Dung khi Colab bi ngat runtime. Cell nay tiep tuc tu `last.pt`.


In [ ]:
RESUME = OUTPUT_DIR / 'last.pt'
if not RESUME.exists():
    raise FileNotFoundError(f'Chua co checkpoint de resume: {RESUME}. Hay train thanh cong truoc.')

data_arg = build_data_arg()
!python -m src.model.train {data_arg} \
  --output-dir {OUTPUT_DIR} \
  --image-size 384 \
  --batch-size 8 \
  --epochs 20 \
  --grad-accum 2 \
  --nec 10 \
  --amp fp16 \
  --num-workers 2 \
  --resume {RESUME}


## 11. Inference sample
Doi `IMAGE_PATH` sang anh ban muon test.


In [ ]:
IMAGE_PATH = Path('/content/drive/MyDrive/sample.jpg')
CHECKPOINT = OUTPUT_DIR / 'best.pt'
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}. Hay train thanh cong truoc.')
!python -m src.model.inference --image {IMAGE_PATH} --checkpoint {CHECKPOINT} --nec 10
